In [1]:
import pandas as pd
import numpy as np
import pingouin as pg
from semopy import Model
from semopy.stats import calc_stats

In [2]:
#load the dataset 

In [3]:
try:
    df_mapped = pd.read_csv("clean_data61-80.csv")
    print(f"Dataset successfully synchronized. Validated records: {len(df_mapped)}")
except FileNotFoundError:
    print("Error: Please ensure 'cleaner_data.csv' is saved in this exact directory folder.")

construct_items = ['PPV1', 'PPV2', 'PPV3', 'PPR1', 'PPR2', 'TR1', 'TR2', 'ITU1', 'ITU2']

Dataset successfully synchronized. Validated records: 6


In [4]:
#Verifying the measurement reliablity using crohnbachs alpha 


In [5]:
print("\n" + "="*50)
print("PHASE 1: MEASUREMENT MODEL RELIABILITY (ALPHA)")
print("="*50)
print(f"Alpha - Perceived Personalisation Value (PPV): {pg.cronbach_alpha(data=df_mapped[['PPV1', 'PPV2', 'PPV3']])[0]:.3f}")
print(f"Alpha - Perceived Privacy Risk (PPR):         {pg.cronbach_alpha(data=df_mapped[['PPR1', 'PPR2']])[0]:.3f}")
print(f"Alpha - Trust (TR):                           {pg.cronbach_alpha(data=df_mapped[['TR1', 'TR2']])[0]:.3f}")
print(f"Alpha - Intention to Use (ITU):               {pg.cronbach_alpha(data=df_mapped[['ITU1', 'ITU2']])[0]:.3f}")


PHASE 1: MEASUREMENT MODEL RELIABILITY (ALPHA)
Alpha - Perceived Personalisation Value (PPV): 0.954
Alpha - Perceived Privacy Risk (PPR):         0.629
Alpha - Trust (TR):                           0.909
Alpha - Intention to Use (ITU):               0.429


In [6]:
#model structure and fit 

In [7]:
sem_specification = """
# Measurement Model Equations
PPV =~ PPV1 + PPV2 + PPV3
PPR =~ PPR1 + PPR2
TR  =~ TR1 + TR2
ITU =~ ITU1 + ITU2

# Structural Model Path Equations
TR  ~ PPV + PPR
ITU ~ PPV + PPR + TR
"""

print("\n" + "="*50)
print("PHASE 2: STRUCTURAL MODEL PATH REGRESSION")
print("="*50)

model = Model(sem_specification)
model.fit(df_mapped)


PHASE 2: STRUCTURAL MODEL PATH REGRESSION


SolverResult(fun=np.float64(7.989254944797532), success=True, n_it=65, x=array([ 6.74886604e-01,  7.89671585e-01,  2.04589757e+00,  6.19571690e-01,
        5.52466144e+00, -7.39904002e-01, -4.18507541e+00, -6.72595030e-02,
       -5.19799005e-02,  1.67706942e-01,  4.37106503e-03,  1.92157513e-01,
        8.13327574e-02,  2.04939858e-01, -4.78506516e-01,  1.73171635e-02,
        6.14182920e-01,  1.29208390e+00,  1.80424641e-01,  2.17103043e-01,
        8.98419177e-13,  4.65857551e-13,  9.13008139e-13,  1.54889304e-01]), message='Optimization terminated successfully', name_method='SLSQP', name_obj='MLW')

In [8]:
#regression coefficients 


In [9]:
results = model.inspect()
structural_paths = results[results['op'] == '~'].copy()

In [10]:
#Standard model path coefficents 

In [11]:
structural_paths['Beta'] = structural_paths['Estimate']
print(structural_paths[['lval', 'op', 'rval', 'Beta', 'z-value', 'p-value']])

    lval op rval      Beta   z-value   p-value
0     TR  ~  PPV -0.739904 -0.587754  0.556697
1     TR  ~  PPR -4.185075 -1.248701  0.211775
2    ITU  ~  PPV -0.067260 -0.754033   0.45083
3    ITU  ~  PPR -0.051980 -1.523783  0.127563
4    ITU  ~   TR  0.167707  0.842481  0.399519
5   PPV1  ~  PPV  1.000000         -         -
6   PPV2  ~  PPV  0.674887  3.434883  0.000593
7   PPV3  ~  PPV  0.789672  6.242672       0.0
8   PPR1  ~  PPR  1.000000         -         -
9   PPR2  ~  PPR  2.045898  2.737786  0.006185
10   TR1  ~   TR  1.000000         -         -
11   TR2  ~   TR  0.619572  4.346151  0.000014
12  ITU1  ~  ITU  1.000000         -         -
13  ITU2  ~  ITU  5.524661  0.890118  0.373403


In [12]:
# Global metrics of fit evualtation 


In [13]:
print("\n" + "="*50)
print("PHASE 3: GLOBAL MODEL FIT INDEX EVALUATION")
print("="*50)
stats = calc_stats(model)

# Debug: Print available columns to identify the correct key name
print("Available columns in stats:", stats.columns.tolist())
print("Available index values:", stats.index.tolist())
print("\nStats DataFrame:")
print(stats)

chi2 = stats.loc['Value', 'chi2']

# Try different possible column names for degrees of freedom
try:
    df_deg = stats.loc['Value', 'doF']
except KeyError:
    try:
        df_deg = stats.loc['Value', 'df']  # Alternative name
    except KeyError:
        try:
            df_deg = stats.loc['Value', 'DoF']  # Alternative capitalization
        except KeyError:
            try:
                df_deg = stats.loc['Value', 'degrees_of_freedom']  # Full name
            except KeyError:
                print("Warning: Could not find degrees of freedom column")
                df_deg = 1  # Default value to avoid division by zero

chi2_ratio = chi2 / df_deg if df_deg != 0 else np.nan

print(f"Chi-square/df Ratio (χ²/df): {chi2_ratio:.3f}  (Target: < 3.0)")
print(f"Comparative Fit Index (CFI):  {stats.loc['Value', 'CFI']:.3f}  (Target: > 0.90)")
print(f"Tucker-Lewis Index (TLI):     {stats.loc['Value', 'TLI']:.3f}  (Target: > 0.90)")
print(f"RMSEA:                        {stats.loc['Value', 'RMSEA']:.3f}  (Target: < 0.08)")


PHASE 3: GLOBAL MODEL FIT INDEX EVALUATION
Available columns in stats: ['DoF', 'DoF Baseline', 'chi2', 'chi2 p-value', 'chi2 Baseline', 'CFI', 'GFI', 'AGFI', 'NFI', 'TLI', 'RMSEA', 'AIC', 'BIC', 'LogLik']
Available index values: ['Value']

Stats DataFrame:
       DoF  DoF Baseline      chi2  chi2 p-value  chi2 Baseline       CFI  \
Value   21            36  47.93553      0.000702     113.818669  0.653868   

            GFI      AGFI       NFI       TLI     RMSEA       AIC        BIC  \
Value  0.578843  0.278017  0.578843  0.406631  0.506487  32.02149  27.023717   

         LogLik  
Value  7.989255  
Chi-square/df Ratio (χ²/df): 2.283  (Target: < 3.0)
Comparative Fit Index (CFI):  0.654  (Target: > 0.90)
Tucker-Lewis Index (TLI):     0.407  (Target: > 0.90)
RMSEA:                        0.506  (Target: < 0.08)
